# 01 — Marketing Campaign Data Inspection

This notebook loads the supplied ZIP archive, inventories its contents, reads the five Parquet tables, and performs first-pass structural and data-quality checks. It does not assume row counts or business results in advance.

In [1]:
%pip -q install pyarrow pandas

import zipfile
from pathlib import Path
import pandas as pd
from IPython.display import display

print('Dependencies ready')

Dependencies ready


## 1. Upload the dataset ZIP

Upload the same ZIP archive supplied for this project. Keep the original archive unchanged.

In [2]:
from google.colab import files

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Please upload the project ZIP file.')

archive_path = Path('/content') / zip_name
print(f'Uploaded: {zip_name} | {archive_path.stat().st_size:,} bytes')

Saving Marketing Campaign Performance & ROI Optimization.zip to Marketing Campaign Performance & ROI Optimization.zip
Uploaded: Marketing Campaign Performance & ROI Optimization.zip | 2,723,971 bytes


In [3]:
extract_dir = Path('/content/marketing_campaign_data')
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(archive_path) as zf:
    bad_member = zf.testzip()
    if bad_member:
        raise RuntimeError(f'ZIP integrity check failed at: {bad_member}')
    zf.extractall(extract_dir)
    inventory = pd.DataFrame([
        {'file': info.filename, 'size_bytes': info.file_size}
        for info in zf.infolist() if not info.is_dir()
    ])

display(inventory)
print('ZIP integrity check passed')

,file,size_bytes
0,Marketing Campaign Performance & ROI Optimizat...,26074
1,Marketing Campaign Performance & ROI Optimizat...,8368
2,Marketing Campaign Performance & ROI Optimizat...,763
3,Marketing Campaign Performance & ROI Optimizat...,530504
4,Marketing Campaign Performance & ROI Optimizat...,2202980
5,Marketing Campaign Performance & ROI Optimizat...,2279


ZIP integrity check passed


## 2. Load Parquet tables

The SQL schema declares five tables. This cell reads the actual files and reports observed row counts and columns.

In [4]:
parquet_paths = {p.stem: p for p in extract_dir.rglob('*.parquet')}
expected_tables = {'channels', 'campaigns', 'ad_groups', 'daily_performance', 'conversions'}
missing_tables = expected_tables - set(parquet_paths)
if missing_tables:
    raise FileNotFoundError(f'Missing expected Parquet tables: {sorted(missing_tables)}')

tables = {name: pd.read_parquet(path) for name, path in parquet_paths.items()}
table_summary = pd.DataFrame([
    {'table': name, 'rows': len(df), 'columns': len(df.columns), 'memory_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 3)}
    for name, df in tables.items()
]).sort_values('table').reset_index(drop=True)

display(table_summary)
for name, df in tables.items():
    print(f'\\n--- {name} | shape={df.shape} ---')
    print(df.dtypes.to_string())
    display(df.head(3))

,table,rows,columns,memory_mb
0,ad_groups,3000,5,0.535
1,campaigns,600,7,0.193
2,channels,9,4,0.001
3,conversions,40000,8,8.063
4,daily_performance,260000,8,78.354


\n--- channels | shape=(9, 4) ---
channel_id       int32
channel_name    object
medium          object
is_paid           bool


,channel_id,channel_name,medium,is_paid
0,1,Paid Search,cpc,True
1,2,Paid Social,cpc,True
2,3,Display,cpm,True


\n--- campaigns | shape=(600, 7) ---
campaign_id       int32
campaign_name    object
channel_id        int32
objective        object
started_on       object
ended_on         object
budget           object


,campaign_id,campaign_name,channel_id,objective,started_on,ended_on,budget
0,1,Spring Launch UK,4,retention,2025-07-26,None,81100.00
1,2,Summer Sale UK,2,traffic,2024-03-13,2024-09-08,25000.00
2,3,Back to School UK,8,awareness,2025-10-05,2026-01-11,47700.00


\n--- ad_groups | shape=(3000, 5) ---
ad_group_id         int32
campaign_id         int32
ad_group_name      object
target_audience    object
status             object


,ad_group_id,campaign_id,ad_group_name,target_audience,status
0,1,59,AG-00001 in_market,in_market,active
1,2,1,AG-00002 broad,broad,active
2,3,46,AG-00003 remarketing,remarketing,archived


\n--- daily_performance | shape=(260000, 8) ---
performance_id       int32
ad_group_id          int32
activity_date       object
impressions          int32
clicks               int32
spend               object
conversions          int32
conversion_value    object


,performance_id,ad_group_id,activity_date,impressions,clicks,spend,conversions,conversion_value
0,1,2747,2024-01-09,37,2,4.99,0,0.00
1,2,2747,2024-01-10,0,0,0.00,0,0.00
2,3,2747,2024-01-11,72,4,1.98,0,0.00


\n--- conversions | shape=(40000, 8) ---
conversion_id                      int32
ad_group_id                        int32
first_touch_channel_id             int32
last_touch_channel_id              int32
converted_at              datetime64[us]
revenue                           object
touch_count                        int32
device                            object


,conversion_id,ad_group_id,first_touch_channel_id,last_touch_channel_id,converted_at,revenue,touch_count,device
0,1,1972,3,2,2025-02-01 18:54:57,59.49,5,desktop
1,2,1049,7,7,2024-12-20 13:10:19,133.62,2,tablet
2,3,1675,7,7,2024-02-13 17:02:00,30.26,3,mobile


## 3. First-pass data-quality checks

These checks are diagnostic only. Do not remove records until the reason and business impact are understood.

In [5]:
quality_rows = []
for name, df in tables.items():
    quality_rows.append({
        'table': name,
        'rows': len(df),
        'duplicate_rows': int(df.duplicated().sum()),
        'total_missing_cells': int(df.isna().sum().sum()),
        'columns_with_missing': int(df.isna().any().sum())
    })
quality_summary = pd.DataFrame(quality_rows).sort_values('table').reset_index(drop=True)
display(quality_summary)

,table,rows,duplicate_rows,total_missing_cells,columns_with_missing
0,ad_groups,3000,0,0,0
1,campaigns,600,0,127,1
2,channels,9,0,0,0
3,conversions,40000,0,0,0
4,daily_performance,260000,0,0,0


In [6]:
# Key uniqueness and relationship checks based on the supplied SQL schema
primary_keys = {
    'channels': 'channel_id',
    'campaigns': 'campaign_id',
    'ad_groups': 'ad_group_id',
    'daily_performance': 'performance_id',
    'conversions': 'conversion_id'
}

key_checks = []
for table_name, key in primary_keys.items():
    series = tables[table_name][key]
    key_checks.append({
        'table': table_name,
        'primary_key': key,
        'null_keys': int(series.isna().sum()),
        'duplicate_key_values': int(series.duplicated().sum())
    })
display(pd.DataFrame(key_checks))

foreign_keys = [
    ('campaigns', 'channel_id', 'channels', 'channel_id'),
    ('ad_groups', 'campaign_id', 'campaigns', 'campaign_id'),
    ('daily_performance', 'ad_group_id', 'ad_groups', 'ad_group_id'),
    ('conversions', 'ad_group_id', 'ad_groups', 'ad_group_id'),
    ('conversions', 'first_touch_channel_id', 'channels', 'channel_id'),
    ('conversions', 'last_touch_channel_id', 'channels', 'channel_id')
]

fk_checks = []
for child, child_col, parent, parent_col in foreign_keys:
    child_values = tables[child][child_col].dropna()
    parent_values = set(tables[parent][parent_col].dropna().unique())
    unmatched = int((~child_values.isin(parent_values)).sum())
    fk_checks.append({'relationship': f'{child}.{child_col} -> {parent}.{parent_col}', 'unmatched_child_rows': unmatched})
display(pd.DataFrame(fk_checks))

,table,primary_key,null_keys,duplicate_key_values
0,channels,channel_id,0,0
1,campaigns,campaign_id,0,0
2,ad_groups,ad_group_id,0,0
3,daily_performance,performance_id,0,0
4,conversions,conversion_id,0,0


,relationship,unmatched_child_rows
0,campaigns.channel_id -> channels.channel_id,0
1,ad_groups.campaign_id -> campaigns.campaign_id,0
2,daily_performance.ad_group_id -> ad_groups.ad_...,0
3,conversions.ad_group_id -> ad_groups.ad_group_id,0
4,conversions.first_touch_channel_id -> channels...,0
5,conversions.last_touch_channel_id -> channels....,0


## 4. Save inspection outputs

Export small summary files for reproducibility. Do not export or commit the raw tables from this notebook.

## 4. Review campaign-level missing values

Inspect campaigns with incomplete metadata before deciding whether any records need correction or exclusion.

# Campaign-level missing-value review

missing_by_column = tables['campaigns'].isna().sum()
missing_campaigns = tables['campaigns'][tables['campaigns'].isna().any(axis=1)]

print('Missing values by campaign column:')
display(missing_by_column[missing_by_column > 0])
print(f'Campaign records with at least one missing value: {len(missing_campaigns)}')
display(missing_campaigns.head(10))

## 5. Save inspection outputs

Export compact diagnostic summaries for reproducibility. Do not export raw tables from this notebook.

output_dir = Path('/content/inspection_outputs')
output_dir.mkdir(exist_ok=True)

table_summary.to_csv(output_dir / 'table_summary.csv', index=False)
quality_summary.to_csv(output_dir / 'data_quality_summary.csv', index=False)
pd.DataFrame(key_checks).to_csv(output_dir / 'primary_key_checks.csv', index=False)
pd.DataFrame(fk_checks).to_csv(output_dir / 'foreign_key_checks.csv', index=False)
missing_by_column[missing_by_column > 0].rename('missing_values').to_csv(output_dir / 'campaign_missing_values.csv')

print(f'Inspection summaries saved to: {output_dir}')

In [7]:
# Campaign-level missing-value review

missing_by_column = tables['campaigns'].isna().sum()
missing_campaigns = tables['campaigns'][tables['campaigns'].isna().any(axis=1)]

print('Missing values by campaign column:')
display(missing_by_column[missing_by_column > 0])
print(f'Campaign records with at least one missing value: {len(missing_campaigns)}')
display(missing_campaigns.head(10))

Missing Values by Column:


,0
ended_on,127


First 10 Affected Campaigns:


,campaign_id,campaign_name,channel_id,objective,started_on,ended_on,budget
0,1,Spring Launch UK,4,retention,2025-07-26,None,81100.00
7,8,Product Launch UK,9,awareness,2025-06-06,None,5400.00
12,13,Spring Launch US,5,conversions,2024-01-25,None,6800.00
16,17,Winter Clearance US,5,awareness,2025-04-23,None,15700.00
18,19,Brand Always On US,3,traffic,2024-01-06,None,390800.00
30,31,Brand Always On DE,4,conversions,2025-04-10,None,27600.00
35,36,Competitor Conquest DE,9,retention,2024-01-24,None,5500.00
42,43,Brand Always On FR,7,awareness,2024-08-09,None,7000.00
43,44,Product Launch FR,8,conversions,2024-11-15,None,16000.00
52,53,Winter Clearance ES,1,conversions,2024-05-22,None,12600.00


Total Affected Campaigns: 127
